# Level 2: metric views

Creates YAML 1.1 metric views for line performance, quality, and maintenance. OEE is a ratio of sums, not an average of ratios. Query measures with `MEASURE()` only.


## Bind config


In [ ]:
import sys
from pathlib import Path

dbutils.widgets.text("catalog", "YOUR_CATALOG", "Catalog")
dbutils.widgets.text("schema", "YOUR_SCHEMA", "Schema")
dbutils.widgets.text("warehouse_id", "YOUR_WAREHOUSE_ID", "SQL warehouse id")
dbutils.widgets.text("telemetry_rows", "10000000", "Telemetry rows (10000000 or 50000000)")
dbutils.widgets.text("repo_root", "", "Workspace folder that contains shared/")

here = Path.cwd().resolve()
root = None
repo_root_w = dbutils.widgets.get("repo_root").strip()
if repo_root_w and (Path(repo_root_w) / "shared" / "config.py").exists():
    root = Path(repo_root_w)
if root is None:
    for candidate in [here, *here.parents]:
        if (candidate / "shared" / "config.py").exists():
            root = candidate
            break
if root is None:
    raise FileNotFoundError("shared/config.py not found. Set repo_root to the workspace folder that contains shared/.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from shared.config import load_config
from shared.lib.genie_client import GenieClient

cfg = load_config()
print("catalog", cfg.catalog)
print("schema", cfg.schema)
print("fqn", cfg.fqn)


## Deploy the three metric views

YAML files live in `metric_views/`. Catalog and schema are substituted at run time.


In [ ]:
from pathlib import Path

mv_dir = root / "level-2-performance-lab" / "metric_views"
mapping = {
    "mv_line_performance.yaml": "mv_line_performance",
    "mv_quality.yaml": "mv_quality",
    "mv_maintenance.yaml": "mv_maintenance",
}
for file_name, view_name in mapping.items():
    yaml_text = (mv_dir / file_name).read_text().replace("${catalog}", cfg.catalog).replace("${schema}", cfg.schema)
    spark.sql(
        f"""CREATE OR REPLACE VIEW {cfg.table(view_name)}
WITH METRICS
LANGUAGE YAML
AS $$
{yaml_text}
$$"""
    )
    print("created", cfg.table(view_name))

oee = spark.sql(
    f"SELECT MEASURE(`oee`) AS oee FROM {cfg.table('mv_line_performance')}"
).collect()[0]["oee"]
indep = spark.sql(
    f"""SELECT SUM(good_units * ideal_cycle_seconds) / NULLIF(SUM(planned_seconds), 0) AS oee
    FROM {cfg.table('fct_line_shift')}"""
).collect()[0]["oee"]
print("MEASURE oee", oee, "independent", indep)
assert abs(float(oee) - float(indep)) < 1e-9
print("PASS OEE ratio of sums")


## Optional materialization (Public Preview)

Skip this cell if row filters or masks are in play. Materialization cannot be used with those.


In [ ]:
print("Optional. In the UI, open the metric view, set a refresh schedule, and compare latency in notebook 14.")
print("If ALTER MATERIALIZATION fails, use the UI fallback. Do not combine with RLS, masks, or ABAC.")
